<div style="padding: 1em 0.5em; color: #fff; background-color: #0969da; font-size: 1.2em;">
    Jour 2 — Embeddings Evo2 + classifieur (le « teacher »)
</div>
<div style="border-left: 2px solid #0969da; min-height: 1.5em;margin-left: 1em;padding: 1em;">
    - Charger les embeddings Evo2 pré-calculés et visualiser l'espace de représentation (PCA) avant tout entraînement<br>
    - Entraîner une petite tête MLP sur ces embeddings gelés : c'est notre modèle « teacher »<br>
    - Mesurer le gain de précision face aux références du Jour 1, et son coût (dépendance à un modèle de 7 milliards de paramètres)<br>
</div>

#### **Votre identité**

Double-cliquez sur cette cellule et complétez, puis exécutez-la (`Maj + Entrée`).

- **Nom & prénom :** _à compléter_
- **Groupe / binôme :** _à compléter_
- **Date :** _à compléter_

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

**Evo2** est un grand modèle de fondation pour l'ADN (architecture StripedHyena2) entraîné
sur des génomes à travers l'arbre du vivant. Plutôt que de concevoir des caractéristiques à
la main (k-mers) ou de les apprendre à partir de zéro sur notre petit jeu de données (le
CNN), on peut demander à un modèle qui a déjà « lu » une énorme quantité d'ADN de
représenter nos fenêtres pour nous.

Nous accédons à Evo2 via l'**API hébergée par NVIDIA** plutôt qu'en exécutant nous-mêmes
le modèle à 7 milliards de paramètres. Les organisateurs ont déjà appelé l'API sur chaque
fenêtre étiquetée issue de `01_kmer_and_cnn_baselines.ipynb` et sauvegardé les résultats —
c'est le plus grand levier de faisabilité de la semaine : vous n'attendez jamais un appel
API en direct.

<img src="https://raw.githubusercontent.com/Genereux-akotenou/EEIA-bioAI-Workshop-project/main/day2/assets/pretrained_paradigm.png"/>

---

Le modèle de fondation que nous utiliserons ici est **Evo2**. Il a été pré-entraîné sur de
grandes quantités de séquences d'ADN issues de génomes de tout l'arbre du vivant. L'idée est
la même que celle de ChatGPT — apprendre à prédire la suite d'une séquence — mais appliquée
au langage du vivant plutôt qu'au texte.

Pour les plus curieux, voici quelques ressources complémentaires. **Leur lecture n'est pas
nécessaire pour atteindre les objectifs de la journée.**

- [Evo2 expliqué en vidéo](https://www.youtube.com/watch?v=o_w-E--u7GQ)
- [Le dépôt GitHub d'Evo2](https://github.com/arcinstitute/evo2)
- [La page officielle du projet (Arc Institute)](https://arcinstitute.org/tools/evo)


In [ ]:
import sys
sys.path.append("src")

import torch
from embeddings import load_supervised_embeddings
from models.classifier_heads import MLPHead
from eval import evaluate_logits, count_params, measure_latency_torch
from viz import plot_embedding_space

EMB_DIR = "../2-data/embeddings"

# TODO : chargez les embeddings pré-calculés train/val avec load_supervised_embeddings(EMB_DIR, ...)
X_train, y_train, ids_train = ...
X_val, y_val, ids_val = ...
print("embedding dim:", X_train.shape[1], "| train windows:", X_train.shape[0])

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Visualisons l'espace des embeddings avant d'entraîner quoi que ce soit**

In [ ]:
# TODO : appelez plot_embedding_space sur les embeddings d'entraînement (method="pca")
# avec un titre du type "Evo2 embeddings, colored by coding/non-coding label"
...

#### **Entraînons le teacher : une petite tête MLP sur les embeddings gelés**

In [ ]:
# N_EPOCHS : même budget d'entraînement pour TOUS les modèles de la semaine,
# pour que la comparaison du Jour 4 porte sur la représentation et non sur
# la durée d'entraînement. Modifiable ici.
N_EPOCHS = 100

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)

# TODO : instanciez MLPHead(d_in=X_train.shape[1]) et un optimiseur Adam (lr=1e-3)
teacher = ...
optimizer = ...

n = X_train_t.shape[0]
for epoch in range(N_EPOCHS):
    perm = torch.randperm(n)
    epoch_loss = 0.0
    for start in range(0, n, 256):
        idx = perm[start:start + 256]
        optimizer.zero_grad()
        # TODO : logits du teacher sur ce mini-lot, puis perte BCE-with-logits
        logits = ...
        loss = ...
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    if (epoch + 1) % 5 == 0:
        print(f"epoch {epoch+1}/{N_EPOCHS}: loss={epoch_loss/n:.4f}")

In [ ]:
# TODO : évaluez le teacher sur validation (sans gradient), puis calculez metrics/params/latence
with torch.no_grad():
    val_logits = ...
teacher_metrics = ...
print("Evo2 embeddings + MLP teacher:", teacher_metrics)
print("params:", ..., "| latency (ms/sample):", ...)

#### **Sauvegardons le teacher pour le Jour 3**

La distillation du Jour 3 a besoin de *ce* teacher entraîné. On enregistre ses poids sur
disque pour ne pas avoir à le ré-entraîner demain.

In [ ]:
from pathlib import Path

# poids + dimension d'entrée (nécessaire pour reconstruire l'architecture au Jour 3)
MODEL_DIR = Path("../2-data/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
torch.save({"state_dict": teacher.state_dict(), "d_in": X_train.shape[1]},
           MODEL_DIR / "teacher_mlp.pt")
print("teacher sauvegardé ->", MODEL_DIR / "teacher_mlp.pt")

#### **Point de contrôle**

Comparez `teacher_metrics` au tableau `comparison` du Jour 1 — les embeddings Evo2
devraient donner un gain de précision visible, et le graphique PCA devrait déjà montrer
une certaine séparation des clusters par étiquette, même si les embeddings n'ont jamais
été affinés pour cette tâche.

Les poids du teacher sont maintenant dans `2-data/models/teacher_mlp.pt` : le notebook 03
les rechargera directement. Ses prédictions y deviendront les cibles douces (*soft targets*)
de la distillation.

Suite : `03_knowledge_distillation.ipynb`.

*Bloqué ? La version complète est dans `solution/02_evo2_embeddings_and_classifier.ipynb`.*

<div style="margin-top: 3em;">
  <div style="height: 3px; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>
  <div style="display: flex; align-items: center; gap: 0.9em; padding: 1.1em 1em; font-size: 0.9em; color: #57606a; background-color: #f2f6fd; border-radius: 0 0 6px 6px;">
    <svg width="34" height="34" viewBox="0 0 34 34" fill="none" style="flex: 0 0 auto;">
      <path d="M9 3c0 7 16 7 16 14S9 24 9 31" stroke="#0969da" stroke-width="2" stroke-linecap="round"/>
      <path d="M25 3c0 7-16 7-16 14s16 7 16 14" stroke="#0969da" stroke-width="2" stroke-linecap="round" opacity="0.45"/>
      <circle cx="17" cy="10" r="1.8" fill="#0969da"/>
      <circle cx="17" cy="24" r="1.8" fill="#0969da"/>
    </svg>
    <div style="flex: 1 1 auto;">
      <div style="color: #0969da; font-weight: 600; letter-spacing: 0.03em;">Fin du Jour 2</div>
      <div>Prochaine &eacute;tape &rarr; <code>day3/03_knowledge_distillation.ipynb</code></div>
    </div>
    <div style="flex: 0 0 auto; text-align: right; border-right: 2px solid #0969da; padding-right: 0.9em;">
      <div style="font-weight: 600; color: #24292f;">EEIA &middot; bioAI Workshop</div>
      <div style="font-size: 0.85em;">Semaine 4 &mdash; De l'ADN aux mod&egrave;les compress&eacute;s</div>
    </div>
  </div>
</div>